# Data audit

Check input availability, the released training schema, and the reference data
behind county denominators. This notebook reads training outcomes only. It does
not rebuild the large raw cache or inspect held-out outage outcomes.

Run preparation separately with `python -m outage_prediction.dataset prepare`.
Dataset definitions and provenance are in [the dataset guide](../docs/dataset.md).


In [1]:
from IPython.display import display
from outage_prediction.config import ProjectPaths
from outage_prediction.analysis import audit_sources
from outage_prediction.dataset import load_references, build_counties

paths = ProjectPaths(offline=True)
audit = audit_sources(paths)
display(audit["source_inventory"])


,source,local,location,bytes
0,data/02_combined/eaglei_outages_2014_2025.csv,False,pinned remote (not read for inventory),NaN
1,data/02_combined/storm_events_details_all_year...,False,pinned remote (not read for inventory),NaN
2,processed/outages_hourly,True,D:\Capstone-Power-Outage-Prediction\data\inter...,NaN
3,processed/counties.parquet,True,D:\Capstone-Power-Outage-Prediction\data\inter...,106924.0
4,processed/model_table/train.parquet,True,D:\Capstone-Power-Outage-Prediction\data\proce...,9686396.0
5,processed/model_table/validation.parquet,True,D:\Capstone-Power-Outage-Prediction\data\proce...,1601759.0
6,processed/model_table/test_features.parquet,True,D:\Capstone-Power-Outage-Prediction\data\proce...,1304975.0


## Is the training release structurally usable?

County/episode keys must be unique and labels binary. Missing intensity fields
often reflect event type, so missingness alone is not a reason to discard a field.
The audit raises an error if the training split violates its required checks.


In [2]:
display(audit["training_years"])
display(audit["training_schema"])


,rows,positive_rate
year,,
2017,25991,0.111539
2018,29276,0.122045
2019,32609,0.123647
2020,27446,0.167456
2021,27711,0.155209
2022,34681,0.140394
2023,30768,0.168487


,column,dtype,missing_pct
0,fips,str,0.000000
1,episode_id,int64,0.000000
2,state_abbr,str,0.000000
3,fema_region,int64,0.000000
4,year,int64,0.000000
5,t0,datetime64[us],0.000000
6,n_events,int64,0.000000
7,n_tstm_wind,int64,0.000000
8,n_hail,int64,0.000000
9,n_tornado,int64,0.000000


## Are county denominators credible?

MCC supplies modeled customers. Missing or implausible values use Census
population times the median customers-per-person ratio. State ratios below half
the national median and county ratios outside 0.1–5 trigger substitution. These
are provisional data-quality rules, not measured customer inventories.

Connecticut planning-region readings are mapped to the legacy eight counties
using population weights. The cleaned pipeline retains capped-reading flags.


In [3]:
references = load_references(paths)
counties = build_counties(references["census"], references["mcc"],
                          references["ct_weights"], references["regions"])
display(counties.groupby("denominator_source").agg(counties=("fips", "size")))
display(counties.loc[counties.denominator_source.eq("census_estimate"),
                     ["fips", "county_name", "state_name", "denominator"]].head(25))


,counties
denominator_source,
census_estimate,130
mcc,3118


,fips,county_name,state_name,denominator
67,02013,Aleutians East Borough,Alaska,1757.0
71,02060,Bristol Bay Borough,Alaska,429.0
72,02063,Chugach Census Area,Alaska,3437.0
73,02066,Copper River Census Area,Alaska,1358.0
78,02105,Hoonah-Angoon Census Area,Alaska,1149.0
84,02164,Lake and Peninsula Borough,Alaska,676.0
110,04023,Santa Cruz County,Arizona,24961.0
240,06105,Trinity County,California,7957.0
282,08071,Las Animas County,Colorado,7286.0
294,08095,Phillips County,Colorado,2273.0


## What this audit cannot establish

A missing outage record can mean zero outages or no collection. The observation
flag combines county activity in the year, at least 90% national snapshots on a
day, and the absence of an all-null county-day. It does not establish continuous
county monitoring. Historical all-year source exploration must not be repeated
as a held-out outcome analysis.

No files are exported by default. To export these particular audit tables, call
`export_results(audit, directory=...)` explicitly.

For optional raw source-quality checks, explicitly call `analysis.audit_raw(paths)`.
This can build the large cache; its outage/storm-type summaries use training years only,
while all-year inventory reports counts without held-out outcome analysis.
